## Kinematic Feature Extraction

####  *AUTHOR:* Ehsan Farahbakhsh
####  *CONTACT:* e.farahbakhsh@sydney.edu.au

In [1]:
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from gplately import PlateModelManager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

from lib.main import *
from lib.plot import *

from parameters import parameters

In [2]:
# Timespan for analysis
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

plate_model_name = parameters["plate_model_name"]
anchor_plate_id = parameters["anchor_plate_id"]

plate_model_dir = parameters["plate_model_dir"]
outputs_dir = parameters["outputs_dir"]
feat_maps_dir = parameters["feat_maps_dir"]

if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

feat_maps_dir = os.path.join(outputs_dir, feat_maps_dir)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

n_jobs = 20

In [3]:
# heidari26
rotation_model = plate_model_dir+"/CombinedRotations.rot"

topology_features = [
    plate_model_dir+"/Deforming_Networks_Active.gpml",
    plate_model_dir+"/Deforming_Networks_Inactive.gpml",
    plate_model_dir+"/Feature_Geometries.gpml",
    plate_model_dir+"/Iran-Afghan_Topological Points.gpml",
    plate_model_dir+"/Iran-Afghan-Turan_Topological Line.gpml",
    plate_model_dir+"/Plate_Boundaries.gpml",
    plate_model_dir+"/SA_Iran_Eurasia_Deformation Network.gpml",
    plate_model_dir+"/Sabzevar-Sistan_Topological Line.gpml",
    plate_model_dir+"/SCI_Subduction_points.gpml",
    plate_model_dir+"/SCI_Subduction_Topological_lines.gpml",
    plate_model_dir+"/Topological_lines.gpml",
    plate_model_dir+"/Topological_Points.gpml",
]

static_polygons = plate_model_dir+"/Global_EarthByte_GPlates_PresentDay_StaticPlatePolygons.shp"
coastlines = plate_model_dir+"/Global_coastlines_low_res.shp"
continents = plate_model_dir+"/Global_EarthByte_GPlates_PresentDay_ContinentalPolygons_edited 20.10.2025.gpml"
# continents = plate_model_dir+"/Global_EarthByte_GPlates_PresentDay_ContinentalPolygons.gpml"
COBs = plate_model_dir+"/Global_EarthByte_GeeK07_COB_Terranes_edited.gpml"

In [ ]:
# # PMM
# pmm = PlateModelManager()
# pm = pmm.get_model(plate_model_name)

# rotation_model = pm.get_rotation_model()
# topology_features = pm.get_topologies()

# static_polygons = pm.get_static_polygons()
# coastlines = pm.get_coastlines()
# continents = pm.get_continental_polygons()
# COBs = pm.get_COBs()

In [4]:
if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    subduction_data = run_calculate_convergence(
        min_time=time_min,
        max_time=time_max,
        temporal_resolution=temporal_resolution,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        anchor_plate_id=anchor_plate_id,
        output_filename=subduction_data_filename,
        n_jobs=n_jobs,
        verbose=True,
    )

In [5]:
nan_only = subduction_data.columns[subduction_data.isna().all()]
constant_non_nan = subduction_data.columns[
    (subduction_data.nunique(dropna=True) == 1) & (~subduction_data.isna().all())
]

if len(nan_only) > 0:
    print("NaN-only:", nan_only.tolist())

if len(constant_non_nan) > 0:
    print("Constant:", constant_non_nan.tolist())

features_to_remove = nan_only.union(constant_non_nan)
if len(features_to_remove) > 0:
    subduction_data = subduction_data.drop(columns=features_to_remove)
    # subduction_data.to_csv(subduction_data_filename, index=False)

# Create list of features available for plotting
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
columns_to_remove = ["lon", "lat", "age (Ma)", "subducting_plate_ID", "trench_plate_ID"]
features_plot = [x for x in features_plot if x not in columns_to_remove]

In [6]:
plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
    anchor_plate_id=anchor_plate_id,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
    anchor_plate_id=anchor_plate_id,
)

# Define map projection (Mollweide provides a good global view)
projection = ccrs.Mollweide(central_longitude=60)

In [7]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    if feature == "convergence_obliquity (degrees)":
        feat_min = -90
        feat_max = 90
    elif feature in [
        "convergence_rate (cm/yr)",
        "convergence_rate_orthogonal (cm/yr)",
        "slab_flux (m^2/yr)",
        "subduction_water_flux_lithosphere (t/m/yr)"
    ]:
        feat_min = 0
        feat_max = subduction_data[feature].quantile(0.99)
    else:
        feat_min = subduction_data[feature].quantile(0.01)
        feat_max = subduction_data[feature].quantile(0.99)
    
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")
    # ax.set_global()
    ax.set_extent([30, 90, 15, 45], crs=ccrs.PlateCarree())

    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)

    gplot.plot_plate_motion_vectors(ax, spacingX=5, spacingY=5, normalise=True, scale=50, alpha=0.1, zorder=3)
    
    feat = ax.scatter(subduction_data_t["lon"], subduction_data_t["lat"], 50, marker=".",
                      c=subduction_data_t[feature], cmap=ccm.hawaii_r, vmin=feat_min, vmax=feat_max,
                      transform=ccrs.PlateCarree(), zorder=4)

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    gplot.plot_trenches(ax, color="black", alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.015, color="black", alpha=0.3, zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)
        
    gl.top_labels=False
    # gl.bottom_labels=False
    gl.right_labels=False
    # gl.left_labels=False    
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    # ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    # ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    # ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    
    cbar_feat = fig.colorbar(feat, shrink=0.4, pad=0.06, orientation="horizontal", extend="both")
    cbar_feat.set_label(format_feature_name(feature), fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)
    
    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="tan", edgecolor="none"),
        trench_handle,
        Line2D([0], [0], color="orangered", lw=2),
        Line2D([0], [0], color="dimgray", lw=2),
    ]
               
    custom_labels = [
        "Continental crust",
        "Subduction zone",
        "Plate boundary",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.2),
              handler_map={trench_handle: HandlerTrenchLine()})
    
    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

In [9]:
if os.path.exists(feat_maps_dir):
    print(f"Feature maps are located in {feat_maps_dir}")
else:
    os.makedirs(feat_maps_dir, exist_ok=True)
       
    generate_feat_maps(
        rotation_model,
        topology_features,
        static_polygons,
        coastlines,
        continents,
        COBs,
        subduction_data,
        projection,
        time_steps,
        feature="convergence_rate (cm/yr)",
        output_dir=feat_maps_dir,
        anchor_plate_id=anchor_plate_id,
        n_jobs=n_jobs
    )
    
    output_filenames = [
        os.path.join(feat_maps_dir, f"feat_map_{t:0.0f}Ma.png")
        for t in time_steps
    ]
    
    output_filename = os.path.join(outputs_dir, "feat_animation.mp4")
    create_animation(
        image_filenames=output_filenames[::-1],
        output_filename=output_filename,
        fps=10,
        bitrate="5000k",
    )

Dispatching tasks: 100%|█████████████████████████████████████████████████████████████| 229/229 [02:41<00:00,  1.42it/s]
